# 04 — Evaluation: 50-case suite

ASHA-AI Plan 2.0 · Role C

Runs every case in [`../../docs/EVAL_CASES.csv`](../../docs/EVAL_CASES.csv) through the full triage pipeline (rules → ML → ESI mapper → safety property = `max(rule_level, esi_level)`) and reports:
- Overall accuracy
- Emergency-bucket recall (**must = 100%**)
- Per-class precision / recall / F1
- Macro-F1
- Confusion matrix
- Rule trigger counts

Writes:
- `ml/eval_results.json` — per-case predictions for inspection
- `ml/metrics.txt` — METHODOLOGY.md-ready results block (paste into §5)


In [ ]:
import sys, pathlib, joblib, json
sys.path.insert(0, str(pathlib.Path('..').resolve()))
from train_and_eval import evaluate, feature_columns, ML
model = joblib.load(ML / 'models' / 'xgboost_v1.pkl')
cols = feature_columns()
results = evaluate(model, cols)
print(f"\nemergency_miss_count={results['emergency_miss_count']}  macro_f1={results['macro_f1']:.3f}")


In [ ]:
import pandas as pd
# Failures table — for iteration
cases_df = pd.DataFrame(results['cases'])
fails = cases_df[(cases_df.match == False) & (cases_df.expected != 'REFUSAL')].copy()
print(f'Total misses: {len(fails)}')
fails[['case_id', 'expected', 'predicted', 'category', 'fired_flags', 'symptoms_text']] if len(fails) else 'no misses'


In [ ]:
# If any ER case is missed, this prints the patch needed (alias miss or rule gap)
er_misses = fails[fails.expected == 'Emergency Room']
if len(er_misses) > 0:
    print('!!! Emergency-miss rate > 0 — must fix before publishing:')
    for _, m in er_misses.iterrows():
        print(f"  case {m.case_id}: predicted={m.predicted!r}  fired_flags={m.fired_flags}  text={m.symptoms_text}")
else:
    print('OK — zero missed emergencies.')
